Load cleaned Silver candidates

In [0]:


from pyspark.sql import functions as F

# Step 1: Define cleaned Silver input path
silver_input_path = (
    "/Volumes/workspace/default/railway_silver/clean"
)

# Step 2: Read validated Delta dataset
silver_df = (
    spark.read
    .format("delta")
    .load(silver_input_path)
)

# Step 3: Check record count
print("Total Silver Records:", silver_df.count())

# Step 4: Check column count
print("Total Columns:", len(silver_df.columns))

# Step 5: Inspect schema
silver_df.printSchema()

# Step 6: Preview records
display(silver_df.limit(10))

Identify and organize transformation columns

In [0]:
from pyspark.sql import functions as F

# Step 1: Define railway business columns
business_columns = [
    "date",
    "train_no",
    "train_name",
    "train_type",
    "seq",
    "station_code",
    "halts_here",
    "sch_arrival",
    "sch_arrival_date",
    "sch_departure",
    "sch_departure_date",
    "act_arrival",
    "act_arrival_date",
    "act_departure",
    "act_departure_date",
    "delay_arrival_min",
    "delay_departure_min",
    "platform",
    "distance_km",
    "lat",
    "lng",
    "departed",
    "reported",
    "reported_at"
]

# Step 2: Define essential pipeline metadata
metadata_columns = [
    "source_file",
    "ingestion_timestamp",
    "pipeline_name"
]

# Step 3: Select required columns
transformation_df = silver_df.select(
    *(business_columns + metadata_columns)
)

# Step 4: Verify records and columns
print("Total Records:", transformation_df.count())
print("Total Columns:", len(transformation_df.columns))

# Step 5: Preview selected data
display(transformation_df.limit(10))

Standardize text columns

In [0]:
from pyspark.sql import functions as F

# Step 1: Standardize train names
# Remove extra spaces and apply title case
transformation_df = transformation_df.withColumn(
    "train_name",
    F.initcap(
        F.trim(F.col("train_name"))
    )
)

# Step 2: Standardize train types
# Remove surrounding spaces and convert to uppercase
transformation_df = transformation_df.withColumn(
    "train_type",
    F.upper(
        F.trim(F.col("train_type"))
    )
)

# Step 3: Standardize station codes
# Example: " ndls " becomes "NDLS"
transformation_df = transformation_df.withColumn(
    "station_code",
    F.upper(
        F.trim(F.col("station_code"))
    )
)

# Step 4: Standardize platform values
# Remove unnecessary surrounding spaces
transformation_df = transformation_df.withColumn(
    "platform",
    F.trim(F.col("platform"))
)

# Step 5: Convert empty strings to NULL
# This prevents blank values from being treated as meaningful data
for column_name in [
    "train_name",
    "train_type",
    "station_code",
    "platform"
]:
    transformation_df = transformation_df.withColumn(
        column_name,
        F.when(
            F.col(column_name) == "",
            F.lit(None)
        ).otherwise(
            F.col(column_name)
        )
    )

# Step 6: Verify record and column counts
print("Total Records:", transformation_df.count())
print("Total Columns:", len(transformation_df.columns))

# Step 7: Preview standardized columns
display(
    transformation_df.select(
        "train_no",
        "train_name",
        "train_type",
        "station_code",
        "platform"
    ).limit(10)
)

Inspect date and time formats

In [0]:

from pyspark.sql import functions as F

# Step 1: Select scheduled and actual date/time columns
datetime_columns = [
    "date",
    "sch_arrival_date",
    "sch_arrival",
    "act_arrival_date",
    "act_arrival",
    "sch_departure_date",
    "sch_departure",
    "act_departure_date",
    "act_departure"
]

# Step 2: Preview the actual date/time values
display(
    transformation_df
    .select(*datetime_columns)
    .limit(15)
)

# Step 3: Check distinct date string formats using samples
for column_name in [
    "sch_arrival_date",
    "act_arrival_date",
    "sch_departure_date",
    "act_departure_date"
]:
    print(f"\nSample values for {column_name}:")

    transformation_df.select(column_name) \
        .where(F.col(column_name).isNotNull()) \
        .distinct() \
        .limit(5) \
        .show(truncate=False)

# Step 4: Inspect time string values
for column_name in [
    "sch_arrival",
    "act_arrival",
    "sch_departure",
    "act_departure"
]:
    print(f"\nSample values for {column_name}:")

    transformation_df.select(column_name) \
        .where(F.col(column_name).isNotNull()) \
        .distinct() \
        .limit(5) \
        .show(truncate=False)

# Step 5: Confirm record count remains unchanged
print("Total Records:", transformation_df.count())

Convert railway date and time strings into timestamps

In [0]:
from pyspark.sql import functions as F

# Step 1: Define the date/time format
# Example: 02 Jan 2025 23:13:00
datetime_format = "dd MMM yyyy HH:mm:ss"

# Step 2: Map each new timestamp to its date and time columns
timestamp_mapping = {
    "scheduled_arrival_ts": (
        "sch_arrival_date", "sch_arrival"
    ),
    "actual_arrival_ts": (
        "act_arrival_date", "act_arrival"
    ),
    "scheduled_departure_ts": (
        "sch_departure_date", "sch_departure"
    ),
    "actual_departure_ts": (
        "act_departure_date", "act_departure"
    )
}

# Step 3: Convert each date/time pair into a timestamp
for new_column, (date_column, time_column) in timestamp_mapping.items():

    transformation_df = transformation_df.withColumn(
        new_column,

        # Combine date and time, then parse the timestamp
        F.try_to_timestamp(
            F.concat_ws(
                " ",
                F.trim(F.col(date_column)),
                F.trim(F.col(time_column))
            ),
            F.lit(datetime_format)
        )
    )

# Step 4: Verify record and column counts
print("Total Records:", transformation_df.count())
print("Total Columns:", len(transformation_df.columns))

# Step 5: Preview converted timestamps
display(
    transformation_df.select(
        "scheduled_arrival_ts",
        "actual_arrival_ts",
        "scheduled_departure_ts",
        "actual_departure_ts"
    ).limit(10)
)

timestamp validation

In [0]:

from pyspark.sql import functions as F

timestamp_checks = {
    "scheduled_arrival_ts": ("sch_arrival_date", "sch_arrival"),
    "actual_arrival_ts": ("act_arrival_date", "act_arrival"),
    "scheduled_departure_ts": ("sch_departure_date", "sch_departure"),
    "actual_departure_ts": ("act_departure_date", "act_departure")
}

for ts_col, (date_col, time_col) in timestamp_checks.items():

    # Check whether both original values are available
    source_present = (
        F.col(date_col).isNotNull() &
        F.col(time_col).isNotNull() &
        (F.trim(F.col(date_col)) != "") &
        (F.trim(F.col(time_col)) != "")
    )

    # Count missing or incomplete source values
    missing_source = transformation_df.filter(
        ~source_present
    ).count()

    # Count unsuccessful timestamp conversions
    parsing_failures = transformation_df.filter(
        source_present & F.col(ts_col).isNull()
    ).count()

    # Count all NULL timestamps
    null_count = transformation_df.filter(
        F.col(ts_col).isNull()
    ).count()

    print(f"{ts_col}:")
    print(f"  Missing Source: {missing_source}")
    print(f"  Parsing Failures: {parsing_failures}")
    print(f"  NULL Timestamps: {null_count}")

Calculate arrival and departure delays

In [0]:

from pyspark.sql import functions as F

# Step 1: Calculate arrival delay in minutes
# Negative values indicate early arrival
transformation_df = transformation_df.withColumn(
    "calculated_arrival_delay_min",

    F.round(
        (
            F.unix_timestamp("actual_arrival_ts")
            - F.unix_timestamp("scheduled_arrival_ts")
        ) / 60.0,
        2
    )
)

# Step 2: Calculate departure delay in minutes
# Negative values indicate early departure
transformation_df = transformation_df.withColumn(
    "calculated_departure_delay_min",

    F.round(
        (
            F.unix_timestamp("actual_departure_ts")
            - F.unix_timestamp("scheduled_departure_ts")
        ) / 60.0,
        2
    )
)

# Step 3: Calculate differences from source delay values
# NULL remains NULL when either value is unavailable
transformation_df = transformation_df.withColumn(
    "arrival_delay_difference_min",

    F.round(
        F.col("calculated_arrival_delay_min")
        - F.col("delay_arrival_min"),
        2
    )
)

transformation_df = transformation_df.withColumn(
    "departure_delay_difference_min",

    F.round(
        F.col("calculated_departure_delay_min")
        - F.col("delay_departure_min"),
        2
    )
)

# Step 4: Verify record and column counts
print("Total Records:", transformation_df.count())
print("Total Columns:", len(transformation_df.columns))

# Step 5: Preview delay calculations
display(
    transformation_df.select(
        "train_no",
        "station_code",
        "delay_arrival_min",
        "calculated_arrival_delay_min",
        "arrival_delay_difference_min",
        "delay_departure_min",
        "calculated_departure_delay_min",
        "departure_delay_difference_min"
    ).limit(10)
)

Validate calculated delays against source delays

In [0]:


from pyspark.sql import functions as F

# Step 1: Define source and calculated delay columns
delay_checks = {
    "Arrival": (
        "delay_arrival_min",
        "calculated_arrival_delay_min"
    ),
    "Departure": (
        "delay_departure_min",
        "calculated_departure_delay_min"
    )
}

# Step 2: Compare source and calculated delays
for delay_type, (source_col, calculated_col) in delay_checks.items():

    # Both delay values are available
    comparable = (
        F.col(source_col).isNotNull() &
        F.col(calculated_col).isNotNull()
    )

    # Both values exist and differ by more than 1 minute
    mismatch = (
        comparable &
        (
            F.abs(
                F.col(source_col) - F.col(calculated_col)
            ) > 1
        )
    )

    # Count comparable records
    comparable_count = transformation_df.filter(
        comparable
    ).count()

    # Count mismatched records
    mismatch_count = transformation_df.filter(
        mismatch
    ).count()

    # Count matching records
    matching_count = comparable_count - mismatch_count

    # Count records missing either delay value
    missing_count = transformation_df.filter(
        ~comparable
    ).count()

    print(f"\n{delay_type} Delay Validation:")
    print("Comparable Records:", comparable_count)
    print("Matching Records:", matching_count)
    print("Mismatched Records:", mismatch_count)
    print("Missing Comparison Values:", missing_count)

Investigate mismatched delay records

In [0]:


from pyspark.sql import functions as F

# Step 1: Identify arrival delay mismatches
arrival_mismatch_df = transformation_df.filter(
    F.col("arrival_delay_difference_min").isNotNull() &
    (F.abs(F.col("arrival_delay_difference_min")) > 1)
)

# Step 2: Identify departure delay mismatches
departure_mismatch_df = transformation_df.filter(
    F.col("departure_delay_difference_min").isNotNull() &
    (F.abs(F.col("departure_delay_difference_min")) > 1)
)

# Step 3: Display mismatch counts
print("Arrival Mismatches:", arrival_mismatch_df.count())
print("Departure Mismatches:", departure_mismatch_df.count())

# Step 4: Inspect arrival discrepancies
print("\nArrival Mismatch Details:")

display(
    arrival_mismatch_df.select(
        "date",
        "train_no",
        "station_code",
        "scheduled_arrival_ts",
        "actual_arrival_ts",
        "delay_arrival_min",
        "calculated_arrival_delay_min",
        "arrival_delay_difference_min"
    ).orderBy(
        F.desc(F.abs(F.col("arrival_delay_difference_min")))
    ).limit(10)
)

# Step 5: Inspect departure discrepancies
print("\nDeparture Mismatch Details:")

display(
    departure_mismatch_df.select(
        "date",
        "train_no",
        "station_code",
        "scheduled_departure_ts",
        "actual_departure_ts",
        "delay_departure_min",
        "calculated_departure_delay_min",
        "departure_delay_difference_min"
    ).orderBy(
        F.desc(F.abs(F.col("departure_delay_difference_min")))
    ).limit(10)
)

# Step 6: Summarize discrepancy magnitude
print("\nDeparture Mismatch Statistics:")

departure_mismatch_df.select(
    F.min(
        F.abs(F.col("departure_delay_difference_min"))
    ).alias("min_difference"),

    F.avg(
        F.abs(F.col("departure_delay_difference_min"))
    ).alias("avg_difference"),

    F.max(
        F.abs(F.col("departure_delay_difference_min"))
    ).alias("max_difference")
).show()

Print mismatch details

In [0]:


# Show all 5 arrival mismatches
print("ARRIVAL MISMATCHES")

arrival_mismatch_df.select(
    "train_no",
    "station_code",
    "delay_arrival_min",
    "calculated_arrival_delay_min",
    "arrival_delay_difference_min"
).show(5, truncate=False)

# Show 5 departure mismatches
print("DEPARTURE MISMATCHES")

departure_mismatch_df.select(
    "train_no",
    "station_code",
    "delay_departure_min",
    "calculated_departure_delay_min",
    "departure_delay_difference_min"
).show(5, truncate=False)

# Calculate departure mismatch statistics
print("DEPARTURE MISMATCH STATISTICS")

departure_mismatch_df.select(
    F.min(F.abs("departure_delay_difference_min")).alias("min_difference"),
    F.avg(F.abs("departure_delay_difference_min")).alias("avg_difference"),
    F.max(F.abs("departure_delay_difference_min")).alias("max_difference")
).show()

Create delay quality flags

In [0]:


from pyspark.sql import functions as F

# Step 1: Define the acceptable difference
delay_tolerance = 1.0

# Step 2: Classify arrival delay quality
transformation_df = transformation_df.withColumn(
    "arrival_delay_quality",

    # Cannot compare if either value is missing
    F.when(
        F.col("delay_arrival_min").isNull() |
        F.col("calculated_arrival_delay_min").isNull(),
        F.lit("MISSING_COMPARISON")
    )

    # Flag differences greater than 1 minute
    .when(
        F.abs(F.col("arrival_delay_difference_min"))
        > delay_tolerance,
        F.lit("MISMATCH")
    )

    # Otherwise, the values agree
    .otherwise(F.lit("MATCH"))
)

# Step 3: Classify departure delay quality
transformation_df = transformation_df.withColumn(
    "departure_delay_quality",

    # Cannot compare if either value is missing
    F.when(
        F.col("delay_departure_min").isNull() |
        F.col("calculated_departure_delay_min").isNull(),
        F.lit("MISSING_COMPARISON")
    )

    # Flag differences greater than 1 minute
    .when(
        F.abs(F.col("departure_delay_difference_min"))
        > delay_tolerance,
        F.lit("MISMATCH")
    )

    # Otherwise, the values agree
    .otherwise(F.lit("MATCH"))
)

# Step 4: Verify record and column counts
print("Total Records:", transformation_df.count())
print("Total Columns:", len(transformation_df.columns))

# Step 5: Show arrival quality summary
print("\nArrival Delay Quality:")

transformation_df.groupBy(
    "arrival_delay_quality"
).count().show()

# Step 6: Show departure quality summary
print("\nDeparture Delay Quality:")

transformation_df.groupBy(
    "departure_delay_quality"
).count().show()

Classify railway delays

In [0]:


from pyspark.sql import functions as F

# Step 1: Define a reusable delay classification function
def classify_delay(delay_column):

    # Missing delay cannot be classified
    return (
        F.when(
            F.col(delay_column).isNull(),
            F.lit("UNKNOWN")
        )

        # Negative delay means the train was early
        .when(
            F.col(delay_column) < 0,
            F.lit("EARLY")
        )

        # Delay between 0 and 5 minutes
        .when(
            F.col(delay_column) <= 5,
            F.lit("ON_TIME")
        )

        # Delay greater than 5 and up to 30 minutes
        .when(
            F.col(delay_column) <= 30,
            F.lit("MINOR_DELAY")
        )

        # Delay greater than 30 and up to 60 minutes
        .when(
            F.col(delay_column) <= 60,
            F.lit("MODERATE_DELAY")
        )

        # Delay greater than 60 minutes
        .otherwise(
            F.lit("SEVERE_DELAY")
        )
    )


# Step 2: Create arrival delay category
transformation_df = transformation_df.withColumn(
    "arrival_delay_category",
    classify_delay("calculated_arrival_delay_min")
)


# Step 3: Create departure delay category
transformation_df = transformation_df.withColumn(
    "departure_delay_category",
    classify_delay("calculated_departure_delay_min")
)


# Step 4: Verify records and columns
print("Total Records:", transformation_df.count())
print("Total Columns:", len(transformation_df.columns))


# Step 5: Display arrival delay distribution
print("\nArrival Delay Categories:")

transformation_df.groupBy(
    "arrival_delay_category"
).count().orderBy("arrival_delay_category").show()


# Step 6: Display departure delay distribution
print("\nDeparture Delay Categories:")

transformation_df.groupBy(
    "departure_delay_category"
).count().orderBy("departure_delay_category").show()

Calculate delay recovery at stations

In [0]:


from pyspark.sql import functions as F

# Step 1: Calculate delay recovery in minutes
# Positive = delay reduced
# Negative = delay increased
transformation_df = transformation_df.withColumn(
    "delay_recovery_min",

    F.round(
        F.col("calculated_arrival_delay_min")
        - F.col("calculated_departure_delay_min"),
        2
    )
)

# Step 2: Classify delay recovery
transformation_df = transformation_df.withColumn(
    "delay_recovery_status",

    # Cannot calculate recovery if either delay is missing
    F.when(
        F.col("delay_recovery_min").isNull(),
        F.lit("UNKNOWN")
    )

    # Delay reduced between arrival and departure
    .when(
        F.col("delay_recovery_min") > 0,
        F.lit("RECOVERED")
    )

    # Delay increased between arrival and departure
    .when(
        F.col("delay_recovery_min") < 0,
        F.lit("DELAY_INCREASED")
    )

    # Arrival and departure delays are equal
    .otherwise(
        F.lit("NO_CHANGE")
    )
)

# Step 3: Verify record and column counts
print("Total Records:", transformation_df.count())
print("Total Columns:", len(transformation_df.columns))

# Step 4: Display recovery classification
print("\nDelay Recovery Status:")

transformation_df.groupBy(
    "delay_recovery_status"
).count().orderBy("delay_recovery_status").show()

# Step 5: Preview recovery calculations
display(
    transformation_df.select(
        "train_no",
        "station_code",
        "calculated_arrival_delay_min",
        "calculated_departure_delay_min",
        "delay_recovery_min",
        "delay_recovery_status"
    ).limit(10)
)

Validate operational metrics

In [0]:


from pyspark.sql import functions as F

# Step 1: Check total records
total_records = transformation_df.count()

# Step 2: Check missing delay categories
missing_categories = transformation_df.filter(
    F.col("arrival_delay_category").isNull() |
    F.col("departure_delay_category").isNull() |
    F.col("delay_recovery_status").isNull()
).count()

# Step 3: Check recovery calculation consistency
# Recovery = arrival delay - departure delay
recovery_mismatches = transformation_df.filter(
    F.col("delay_recovery_min").isNotNull() &
    (
        F.abs(
            F.col("delay_recovery_min") -
            (
                F.col("calculated_arrival_delay_min") -
                F.col("calculated_departure_delay_min")
            )
        ) > 0.01
    )
).count()

# Step 4: Check missing arrival timestamps are UNKNOWN
arrival_unknown_errors = transformation_df.filter(
    F.col("actual_arrival_ts").isNull() &
    (F.col("arrival_delay_category") != "UNKNOWN")
).count()

# Step 5: Check missing departure timestamps are UNKNOWN
departure_unknown_errors = transformation_df.filter(
    F.col("actual_departure_ts").isNull() &
    (F.col("departure_delay_category") != "UNKNOWN")
).count()

# Step 6: Check missing recovery values are UNKNOWN
recovery_unknown_errors = transformation_df.filter(
    F.col("delay_recovery_min").isNull() &
    (F.col("delay_recovery_status") != "UNKNOWN")
).count()

# Step 7: Display validation results
print("Total Records:", total_records)
print("Total Columns:", len(transformation_df.columns))
print("Missing Categories:", missing_categories)
print("Recovery Calculation Mismatches:", recovery_mismatches)
print("Arrival UNKNOWN Classification Errors:", arrival_unknown_errors)
print("Departure UNKNOWN Classification Errors:", departure_unknown_errors)
print("Recovery UNKNOWN Classification Errors:", recovery_unknown_errors)

# Step 8: Fail validation if any rule is violated
expected_records = silver_df.count()
assert total_records == expected_records, (
    f"Record count mismatch! Silver clean={expected_records}, transformed={total_records}"
)
assert missing_categories == 0, "Missing delay categories!"
assert recovery_mismatches == 0, "Recovery calculation mismatch!"
assert arrival_unknown_errors == 0, "Arrival classification error!"
assert departure_unknown_errors == 0, "Departure classification error!"
assert recovery_unknown_errors == 0, "Recovery classification error!"

print("OPERATIONAL METRICS VALIDATION: PASSED")

Save transformed Silver data as Delta

In [0]:


from pyspark.sql import functions as F

# Step 1: Define transformed Silver output path
silver_transformed_path = (
    "/Volumes/workspace/default/railway_silver/transformed"
)

# Step 2: Write the transformed DataFrame as Delta
# Overwrite is used because this is a full-refresh pipeline
transformation_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save(silver_transformed_path)

# Step 3: Read the saved Delta dataset independently
saved_silver_df = (
    spark.read
    .format("delta")
    .load(silver_transformed_path)
)

# Step 4: Verify saved records and columns
saved_count = saved_silver_df.count()
saved_columns = len(saved_silver_df.columns)

print("Saved Silver Records:", saved_count)
print("Saved Silver Columns:", saved_columns)

# Step 5: Validate the persisted dataset
assert saved_count == transformation_df.count(), (
    f"FAILED: Silver record count mismatch! Saved={saved_count}, "
    f"expected={transformation_df.count()}"
)

assert saved_silver_df.columns == transformation_df.columns, (
    "FAILED: Transformed Silver schema/column order mismatch!"
)

# Step 6: Confirm successful persistence
print("TRANSFORMED SILVER DELTA SAVE: PASSED")